In [8]:
# --- Cel·la 1: Imports i paths ---
from pathlib import Path
import re
import unicodedata
import math
import random
import pandas as pd

# Paths d'entrada/sortida (canvia si cal)
CSV_PATH = "BASE_COMPLETA_AMPLIADA.csv"
OUT_REPRICED_CSV = "BASE_COMPLETA_AMPLIADA_REPRICED.csv"

# Opcional: normalitzar (només si després exportes a CLIPS, no necessari per re-pricing)
NORMALITZA_INGREDIENTS = False  # millor False per no perdre accents a la detecció per substrings


In [ ]:
# --- Cel·la 2: Utils robustes per parseig CSV ---

def _strip_curly_quotes(s: str) -> str:
    if not isinstance(s, str):
        return s
    return (s.replace("“", '"').replace("”", '"')
             .replace("‘", "'").replace("’", "'")
             .replace("‚", "'").replace("`", "'"))

def _strip_wrapping_quotes(s: str) -> str:
    if not isinstance(s, str):
        return s
    s = s.strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in ("'", '"'):
        return s[1:-1]
    return s

def strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

def robust_split_csv_line(line: str):
    """Separa per comes fora de cometes i fora de [ ... ]"""
    fields, cur = [], []
    in_quotes = False
    quote_char = None
    bracket_depth = 0
    esc = False
    for ch in line:
        if esc:
            cur.append(ch); esc = False; continue
        if ch == "\\":
            esc = True; continue
        if ch in ['"', "'"]:
            if not in_quotes:
                in_quotes = True; quote_char = ch; cur.append(ch)
            else:
                if ch == quote_char:
                    in_quotes = False; quote_char = None; cur.append(ch)
                else:
                    cur.append(ch)
        elif ch == '[' and not in_quotes:
            bracket_depth += 1; cur.append(ch)
        elif ch == ']' and not in_quotes and bracket_depth > 0:
            bracket_depth -= 1; cur.append(ch)
        elif ch == ',' and not in_quotes and bracket_depth == 0:
            fields.append(''.join(cur).strip()); cur = []
        else:
            cur.append(ch)
    fields.append(''.join(cur).strip())
    return fields

def split_top_level_commas_inside_brackets(s: str):
    """Rep un string que comença amb '[' i acaba amb ']' i separa per comes de nivell 1."""
    inner = s.strip()[1:-1]
    items, cur = [], []
    depth_square = depth_round = depth_curly = 0
    esc = False
    for ch in inner:
        if esc:
            cur.append(ch); esc = False; continue
        if ch == "\\":
            esc = True; continue
        if ch == "[":
            depth_square += 1; cur.append(ch); continue
        if ch == "]" and depth_square > 0:
            depth_square -= 1; cur.append(ch); continue
        if ch == "(":
            depth_round += 1; cur.append(ch); continue
        if ch == ")" and depth_round > 0:
            depth_round -= 1; cur.append(ch); continue
        if ch == "{":
            depth_curly += 1; cur.append(ch); continue
        if ch == "}" and depth_curly > 0:
            depth_curly -= 1; cur.append(ch); continue
        if ch == "," and depth_square == depth_round == depth_curly == 0:
            items.append(''.join(cur).strip()); cur = []; continue
        cur.append(ch)
    items.append(''.join(cur).strip())
    cleaned = []
    for t in items:
        t = _strip_curly_quotes(t).strip()
        t = _strip_wrapping_quotes(t).strip()
        if t != "":
            cleaned.append(t)
    return cleaned

def parse_list_cell(s, fallback_empty=False):
    """
    Converteix cel·la a llista:
    - "['a','b']" o ['a','b'] -> ['a','b'] (sense exigir cometes equilibrades)
    - a,b -> ['a','b']
    - val únic -> ['val']
    """
    if s is None:
        return [] if fallback_empty else s
    s = _strip_curly_quotes(str(s)).strip()
    s = _strip_wrapping_quotes(s).strip()
    if s == "":
        return [] if fallback_empty else []
    if s.startswith("[") and s.endswith("]"):
        try:
            return split_top_level_commas_inside_brackets(s)
        except Exception:
            pass
    if "," in s and "[" not in s and "]" not in s:
        return [x.strip().strip("'").strip('"') for x in s.split(",") if x.strip()]
    return [s.strip().strip("'").strip('"')]


In [10]:
# --- Cel·la 2: Utils robustes per parseig CSV ---

def _strip_curly_quotes(s: str) -> str:
    if not isinstance(s, str):
        return s
    return (s.replace("“", '"').replace("”", '"')
             .replace("‘", "'").replace("’", "'")
             .replace("‚", "'").replace("`", "'"))

def _strip_wrapping_quotes(s: str) -> str:
    if not isinstance(s, str):
        return s
    s = s.strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in ("'", '"'):
        return s[1:-1]
    return s

def strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

def robust_split_csv_line(line: str):
    """Separa per comes fora de cometes i fora de [ ... ]"""
    fields, cur = [], []
    in_quotes = False
    quote_char = None
    bracket_depth = 0
    esc = False
    for ch in line:
        if esc:
            cur.append(ch); esc = False; continue
        if ch == "\\":
            esc = True; continue
        if ch in ['"', "'"]:
            if not in_quotes:
                in_quotes = True; quote_char = ch; cur.append(ch)
            else:
                if ch == quote_char:
                    in_quotes = False; quote_char = None; cur.append(ch)
                else:
                    cur.append(ch)
        elif ch == '[' and not in_quotes:
            bracket_depth += 1; cur.append(ch)
        elif ch == ']' and not in_quotes and bracket_depth > 0:
            bracket_depth -= 1; cur.append(ch)
        elif ch == ',' and not in_quotes and bracket_depth == 0:
            fields.append(''.join(cur).strip()); cur = []
        else:
            cur.append(ch)
    fields.append(''.join(cur).strip())
    return fields

def split_top_level_commas_inside_brackets(s: str):
    """Rep un string que comença amb '[' i acaba amb ']' i separa per comes de nivell 1."""
    inner = s.strip()[1:-1]
    items, cur = [], []
    depth_square = depth_round = depth_curly = 0
    esc = False
    for ch in inner:
        if esc:
            cur.append(ch); esc = False; continue
        if ch == "\\":
            esc = True; continue
        if ch == "[":
            depth_square += 1; cur.append(ch); continue
        if ch == "]" and depth_square > 0:
            depth_square -= 1; cur.append(ch); continue
        if ch == "(":
            depth_round += 1; cur.append(ch); continue
        if ch == ")" and depth_round > 0:
            depth_round -= 1; cur.append(ch); continue
        if ch == "{":
            depth_curly += 1; cur.append(ch); continue
        if ch == "}" and depth_curly > 0:
            depth_curly -= 1; cur.append(ch); continue
        if ch == "," and depth_square == depth_round == depth_curly == 0:
            items.append(''.join(cur).strip()); cur = []; continue
        cur.append(ch)
    items.append(''.join(cur).strip())
    cleaned = []
    for t in items:
        t = _strip_curly_quotes(t).strip()
        t = _strip_wrapping_quotes(t).strip()
        if t != "":
            cleaned.append(t)
    return cleaned

def parse_list_cell(s, fallback_empty=False):
    """
    Converteix cel·la a llista:
    - "['a','b']" o ['a','b'] -> ['a','b'] (sense exigir cometes equilibrades)
    - a,b -> ['a','b']
    - val únic -> ['val']
    """
    if s is None:
        return [] if fallback_empty else s
    s = _strip_curly_quotes(str(s)).strip()
    s = _strip_wrapping_quotes(s).strip()
    if s == "":
        return [] if fallback_empty else []
    if s.startswith("[") and s.endswith("]"):
        try:
            return split_top_level_commas_inside_brackets(s)
        except Exception:
            pass
    if "," in s and "[" not in s and "]" not in s:
        return [x.strip().strip("'").strip('"') for x in s.split(",") if x.strip()]
    return [s.strip().strip("'").strip('"')]


In [11]:
# --- Cel·la 3: Carrega CSV robust i construeix DataFrame per re-pricing ---

lines = Path(CSV_PATH).read_text(encoding="utf-8").splitlines()
if not lines:
    raise RuntimeError("El fitxer CSV és buit o no s'ha trobat.")

header_cols = [h.strip() for h in robust_split_csv_line(lines[0])]
lower_map = {h.lower(): i for i, h in enumerate(header_cols)}

def col(name):
    idx = lower_map.get(name.lower())
    if idx is None:
        raise RuntimeError(f"Falta la columna requerida: {name}")
    return idx

# Index dels camps que esperem
IDX_nom   = col("nom_plat")
IDX_mida  = col("mida_racio")
IDX_comp  = col("complexitat")
IDX_temp  = col("temperatura")
IDX_ordre = col("te_ordre")
IDX_form  = col("formalitat")
IDX_preu  = col("preu_cost")
IDX_apte  = col("apte_esdeveniment")
IDX_disp  = col("disponibilitat")
IDX_ing   = col("ingredients")
IDX_cat   = col("categoria")
IDX_proc  = col("procedencia_plat")

data_rows = [robust_split_csv_line(ln) for ln in lines[1:] if ln.strip()]
good_rows = [r for r in data_rows if len(r) >= len(header_cols)]
bad_rows  = [r for r in data_rows if len(r) < len(header_cols)]
if bad_rows:
    print(f"[AVÍS] {len(bad_rows)} files amb menys columnes. S'ignoren.")

def map_te_ordre_raw_to_plain(val: str) -> str:
    v = (val or "").strip().lower()
    if v.startswith("postre"):   return "postres"
    if v.startswith("primer"):   return "primer"
    if v.startswith("segon"):    return "segon"
    # per defecte, tracta com postres
    return "postres"

def norm_complexitat(v: str) -> str:
    v = (v or "").strip().lower()
    return v if v in {"baixa","mitjana","alta"} else "mitjana"

def norm_mida(v: str) -> str:
    v = (v or "").strip().lower()
    return v if v in {"petita","mitjana","gran"} else "mitjana"

def get_cell(r, idx):
    return _strip_curly_quotes(r[idx]) if idx < len(r) else ""

# Construïm registres nets
records = []
for r in good_rows:
    nom = get_cell(r, IDX_nom).strip()
    if not nom:
        continue

    formalitat  = (get_cell(r, IDX_form).strip() or "informal")
    temperatura = (get_cell(r, IDX_temp).strip() or "tebi")
    complexitat = norm_complexitat(get_cell(r, IDX_comp))
    mida_racio  = norm_mida(get_cell(r, IDX_mida))
    te_ordre    = map_te_ordre_raw_to_plain(get_cell(r, IDX_ordre))
    categoria   = (get_cell(r, IDX_cat).strip() or "pastisseria")
    procedencia = (get_cell(r, IDX_proc).strip() or "Internacional")

    # preu en float, acceptant coma decimal
    preu_raw = _strip_wrapping_quotes(get_cell(r, IDX_preu)).replace(",", ".").strip()
    try:
        preu_cost = float(preu_raw)
    except Exception:
        preu_cost = 0.0

    # llistes
    apte_esdeveniment = parse_list_cell(get_cell(r, IDX_apte), fallback_empty=True)
    disponibilitat    = parse_list_cell(get_cell(r, IDX_disp), fallback_empty=True)
    ingredients_list  = parse_list_cell(get_cell(r, IDX_ing),  fallback_empty=True)

    # ingredients normalitzats (opcional)
    if NORMALITZA_INGREDIENTS:
        ing_norm = []
        seen = set()
        for x in ingredients_list:
            tok = normalize_space(x)
            tok = strip_accents(tok).lower()
            tok = tok.replace("'", "").replace("’", "").replace("`", "")
            tok = tok.replace('"', "")
            if tok and tok not in seen:
                seen.add(tok)
                ing_norm.append(tok)
    else:
        ing_norm = ingredients_list[:]  # deixa tal qual

    records.append({
        "nom_plat": nom,
        "formalitat": formalitat,
        "temperatura": temperatura,
        "complexitat": complexitat,
        "mida_racio": mida_racio,
        "te_ordre": te_ordre,                # 'primer' | 'segon' | 'postres'
        "categoria": categoria,              # p.ex. 'carn','peix','pastisseria',...
        "procedencia_plat": procedencia,
        "preu_cost": preu_cost,
        "apte_esdeveniment": apte_esdeveniment,
        "disponibilitat": disponibilitat,
        "ingredients": ingredients_list,     # llista (raw)
        "ingredients_norm": ing_norm,        # llista (potser normalitzada)
    })

df = pd.DataFrame.from_records(records)
print(f"Files carregades: {len(df)}")
df.head(3)


[AVÍS] 2 files amb menys columnes. S'ignoren.
Files carregades: 1108


,nom_plat,formalitat,temperatura,complexitat,mida_racio,te_ordre,categoria,procedencia_plat,preu_cost,apte_esdeveniment,disponibilitat,ingredients,ingredients_norm
0,Amanida Caprese amb mozzarella i alfàbrega,formal,fred,baixa,petita,primer,amanida_fresca,Itàlia,5.8,"[casament, empresa, congres, aniversari]",[estiu],"[tomàquet, mozzarella, alfàbrega, oli d'oliva,...","[tomàquet, mozzarella, alfàbrega, oli d'oliva,..."
1,Amanida Cèsar amb pa torrat i parmesà,formal,fred,baixa,petita,primer,amanida_fresca,Estats Units,5.8,"[casament, empresa, congres, aniversari]","[estiu, primavera, tardor, hivern]","[enciam romà, salsa cèsar, ou, pa torrat, parm...","[enciam romà, salsa cèsar, ou, pa torrat, parm..."
2,Amanida japonesa d'algues amb sèsam torrat,informal,fred,baixa,petita,primer,amanida_fresca,Japó,5.0,"[empresa, congres, aniversari]",[totes],"[algues, ceba, salsa de soja, vinagre d'arròs,...","[algues, ceba, salsa de soja, vinagre d'arròs,..."


In [15]:
# --- Cel·la 4: Regles de re-pricing (versió ≫ més realista i amb més dispersió) ---

RULES = {
    # 1) Categoria: més contrast entre plats “cars” i “lleugers”
    "scale_by_category": {
        # principals “cars”
        "carn": 1.35,
        "marisc": 1.32,
        "peix": 1.28,
        "aus": 1.18,
        "guisat": 1.22,
        "forn_brasa": 1.20,
        # mid-range
        "farinaci_principal": 1.12,
        "vegetal_entrant": 1.08,
        "entrant_fred": 1.05,
        "amanida_fresca": 1.04,
        # postres
        "xoco_intens": 1.14,
        "pastisseria": 1.10,
        "fresc_fruita_gelat": 1.06,
    },

    # 2) te_ordre: segons molt més amunt que primers; postres una mica
    "scale_by_te_ordre": {
        "primer": 1.08,
        "segon": 1.22,
        "postres": 1.10,
    },

    # 3) complexitat: més pes si és “alta”
    "scale_by_complexitat": {
        "alta": 1.15,
        "mitjana": 1.06,
        # "baixa": 1.00
    },

    # 4) formalitat: afegit fix en € (sobretot útil si el cost base és baix)
    "add_by_formalitat": {
        "formal": 1.20,     # abans 0.20
        "informal": 0.00,
    },

    # 5) procedència (substring-insensitive): cuines “premium” pugen més
    "scale_by_procedencia_substr": {
        "Itàlia": 1.05,
        "França": 1.06,
        "Japó": 1.10,
        "Mediterrània": 1.03,
        "Perú": 1.06,
        "Mèxic": 1.04,
        "Nord": 1.05,      # p.ex. escandinava
    },

    # 6) ingredients premium (substring a qualsevol ingredient)
    "scale_by_ingredient_substr": [
        # mar/peix premium
        ("ostres", 1.25), ("llagostí", 1.12), ("carabiner", 1.25),
        ("turbot", 1.18), ("llobarro", 1.12), ("tonyina", 1.10),
        ("salmó", 1.10), ("bacallà", 1.08),
        # carns premium
        ("wagyu", 1.30), ("angus", 1.18), ("vedella", 1.10),
        ("corder", 1.12), ("magret", 1.15), ("foie", 1.15), ("caviar", 1.35),
        ("ibèric", 1.12), ("pernil ibèric", 1.12),
        # tòfones/bolets i especiaria cara
        ("trufa", 1.25), ("tòfona", 1.25), ("ceps", 1.12), ("bolet", 1.08), ("safrà", 1.12),
        # lactis/fruits secs/cacau
        ("formatge", 1.04), ("parmesà", 1.06), ("pistatx", 1.08), ("avellana", 1.06),
        ("xocolata", 1.08), ("xoco", 1.08),
        # tècniques “de luxe”
        ("tàrtar", 1.10), ("carpaccio", 1.10),
    ],

    # 7) jitter: més variabilitat per evitar preus massa agrupats
    "jitter_percent": 14,   # abans 6

    # 8) límits (euros): eixamplem cap amunt i evitem resultats massa baixos
    "clamp_min": 6.50,      # abans 3.00
    "clamp_max": 65.00,     # abans 40.00

    # 9) arrodoniment a cèntims
    "round_cents": True,
}


In [16]:
# --- Cel·la 5: Funcions de re-pricing ---

def _round2(x): 
    return math.floor(x*100+0.5)/100.0

def _apply_scale_where(mask, factor, ser):
    ser.loc[mask] = ser.loc[mask] * factor

def _apply_add_where(mask, delta, ser):
    ser.loc[mask] = ser.loc[mask] + delta

def _contains_substr_robust(lst, needle: str) -> bool:
    """Match robust: sense distingir accents ni majúscules."""
    if not lst:
        return False
    nl = needle.lower()
    nl_na = strip_accents(nl)
    for item in lst:
        s = str(item)
        s1 = s.lower()
        s2 = strip_accents(s1)
        if nl in s1 or nl_na in s2:
            return True
    return False

def repricing(df, rules=RULES, verbose=True):
    out = df.copy()
    price = out["preu_cost"].astype(float)
    base_before = price.copy()

    # 1) per categoria
    for cat, factor in rules.get("scale_by_category", {}).items():
        mask = (out["categoria"].astype(str) == cat)
        _apply_scale_where(mask, factor, price)

    # 2) per te_ordre
    for ord_, factor in rules.get("scale_by_te_ordre", {}).items():
        mask = (out["te_ordre"].astype(str) == ord_)
        _apply_scale_where(mask, factor, price)

    # 3) per complexitat
    for cx, factor in rules.get("scale_by_complexitat", {}).items():
        mask = (out["complexitat"].astype(str) == cx)
        _apply_scale_where(mask, factor, price)

    # 4) per formalitat (increments €)
    for form, delta in rules.get("add_by_formalitat", {}).items():
        mask = (out["formalitat"].astype(str) == form)
        _apply_add_where(mask, float(delta), price)

    # 5) procedència (substring, insensible a accents/majúscules)
    for needle, factor in rules.get("scale_by_procedencia_substr", {}).items():
        nl = needle.lower()
        nl_na = strip_accents(nl)
        col = out["procedencia_plat"].astype(str)
        mask = col.str.lower().str.contains(nl, na=False) | col.apply(lambda s: nl_na in strip_accents(str(s)).lower())
        _apply_scale_where(mask, factor, price)

    # 6) ingredients (substring sobre qualsevol ingredient, robust a accents)
    for needle, factor in rules.get("scale_by_ingredient_substr", []):
        mask = out["ingredients"].apply(lambda lst: _contains_substr_robust(lst, needle))
        _apply_scale_where(mask, factor, price)

    # 7) jitter ±p% (determinista per nom_plat)
    p = float(rules.get("jitter_percent", 0) or 0)
    if p > 0:
        lo, hi = 1.0 - p/100.0, 1.0 + p/100.0
        def _jseed(name):
            return abs(hash(str(name))) % 10_000_000
        new_vals = []
        for name, val in zip(out["nom_plat"], price):
            r = random.Random(_jseed(name)).uniform(lo, hi)
            new_vals.append(val * r)
        price = pd.Series(new_vals, index=price.index)

    # 8) clamp
    cmin = rules.get("clamp_min", None)
    cmax = rules.get("clamp_max", None)
    if cmin is not None:
        price = price.clip(lower=float(cmin))
    if cmax is not None:
        price = price.clip(upper=float(cmax))

    # 9) round
    if rules.get("round_cents", True):
        price = price.apply(_round2)

    out["preu_cost_before"] = base_before.apply(_round2)
    out["preu_cost"] = price

    if verbose:
        diff = (out["preu_cost"] - out["preu_cost_before"])
        print(f"Δ preu_cost — mitjana: {diff.mean():.2f} €, mediana: {diff.median():.2f} €, min: {diff.min():.2f}, max: {diff.max():.2f}")
        print(f"preu_cost abans — min:{out['preu_cost_before'].min():.2f}  P50:{out['preu_cost_before'].median():.2f}  max:{out['preu_cost_before'].max():.2f}")
        print(f"preu_cost després — min:{out['preu_cost'].min():.2f}  P50:{out['preu_cost'].median():.2f}  max:{out['preu_cost'].max():.2f}")
    return out


In [17]:
# --- Cel·la 6: Executa re-pricing i previsualitza ---
df_new = repricing(df, RULES, verbose=True)
df_new.head(5)


Δ preu_cost — mitjana: 4.14 €, mediana: 3.28 €, min: 0.63, max: 22.32
preu_cost abans — min:0.00  P50:6.00  max:15.00
preu_cost després — min:6.50  P50:9.20  max:37.32


,nom_plat,formalitat,temperatura,complexitat,mida_racio,te_ordre,categoria,procedencia_plat,preu_cost,apte_esdeveniment,disponibilitat,ingredients,ingredients_norm,preu_cost_before
0,Amanida Caprese amb mozzarella i alfàbrega,formal,fred,baixa,petita,primer,amanida_fresca,Itàlia,7.26,"[casament, empresa, congres, aniversari]",[estiu],"[tomàquet, mozzarella, alfàbrega, oli d'oliva,...","[tomàquet, mozzarella, alfàbrega, oli d'oliva,...",5.8
1,Amanida Cèsar amb pa torrat i parmesà,formal,fred,baixa,petita,primer,amanida_fresca,Estats Units,7.96,"[casament, empresa, congres, aniversari]","[estiu, primavera, tardor, hivern]","[enciam romà, salsa cèsar, ou, pa torrat, parm...","[enciam romà, salsa cèsar, ou, pa torrat, parm...",5.8
2,Amanida japonesa d'algues amb sèsam torrat,informal,fred,baixa,petita,primer,amanida_fresca,Japó,6.50,"[empresa, congres, aniversari]",[totes],"[algues, ceba, salsa de soja, vinagre d'arròs,...","[algues, ceba, salsa de soja, vinagre d'arròs,...",5.0
3,Amanida de fideus asiàtics amb sèsam,informal,fred,baixa,petita,primer,amanida_fresca,Àsia oriental,6.50,"[empresa, congres, aniversari]",[totes],"[fideus, oli de sèsam, coriandre, ceba tendra,...","[fideus, oli de sèsam, coriandre, ceba tendra,...",5.0
4,Amanida de quinoa i verdures amb salsa de cacauet,informal,fred,baixa,mitjana,primer,amanida_fresca,Perú,7.89,"[empresa, congres, aniversari]",[totes],"[mantega de cacauet, tamari, vinagre d'arròs, ...","[mantega de cacauet, tamari, vinagre d'arròs, ...",6.0


In [18]:
# --- Cel·la 7: Desa resultats ---
# Manté les columnes originals + preu_cost_before i preu_cost (nou)
df_new.to_csv(OUT_REPRICED_CSV, index=False, encoding="utf-8")
print(f"S'ha desat: {Path(OUT_REPRICED_CSV).resolve()}")


S'ha desat: /home/blanca/gia_3/Lab2/BASE_COMPLETA_AMPLIADA_REPRICED.csv


In [ ]:
# --- Conversor CSV -> CLIPS (definstances) amb parseig top-level de llistes ---
# Llegeix 'BASE_COMPLETA_AMPLIADA.csv' i genera 'plats_definitius.clp'

import re
import unicodedata
from pathlib import Path

# ======= Config =======
CSV_PATH = "BASE_COMPLETA_AMPLIADA_REPRICED.csv"
OUT_PATH = "plats_definitius.clp"
CLIPS_BLOCK_NAME = "plats-cataleg"
# --- Normalització d'ingredients ---
NORMALITZA_INGREDIENTS = True  # posa False si vols mantenir els accents

def normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

def normalize_ingredient_token(tok: str) -> str:
    tok = normalize_space(tok)
    # treu accents i posa en minúscules (si ho vols)
    tok = strip_accents(tok).lower()
    # treu apòstrofs i variants tipogràfiques dels ingredients
    tok = tok.replace("'", "").replace("’", "").replace("`", "")
    # opcional: també treure cometes dobles si algun ingredient les porta accidentalment
    tok = tok.replace('"', "")
    return tok


# ======= Utils =======
def _strip_curly_quotes(s: str) -> str:
    if not isinstance(s, str):
        return s
    return (s.replace("“", '"').replace("”", '"')
             .replace("‘", "'").replace("’", "'")
             .replace("‚", "'").replace("`", "'"))

def _strip_wrapping_quotes(s: str) -> str:
    if not isinstance(s, str):
        return s
    s = s.strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in ("'", '"'):
        return s[1:-1]
    return s

def strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def slugify_instance_name(name: str) -> str:
    base = strip_accents(str(name)).lower()
    base = base.replace("&", " i ")
    base = re.sub(r"[^a-z0-9\s\-_\/]", "", base)   # permet _ i /
    base = re.sub(r"[\/]", "-", base)
    base = re.sub(r"\s+", "-", base).strip("-")
    base = re.sub(r"-{2,}", "-", base)
    return base or "plat"

def to_symbol(s: str) -> str:
    return slugify_instance_name(s).replace("-", "_") or "valor"

def sanitize_text(s: str) -> str:
    s = _strip_curly_quotes(str(s)).strip()
    return s.replace('"', '\\"')

def robust_split_csv_line(line: str):
    """Separa a comes fora de cometes i fora de [ ... ]"""
    fields, cur = [], []
    in_quotes = False
    quote_char = None
    bracket_depth = 0
    esc = False
    for ch in line:
        if esc:
            cur.append(ch); esc = False; continue
        if ch == "\\":
            esc = True; continue
        if ch in ['"', "'"]:
            if not in_quotes:
                in_quotes = True; quote_char = ch; cur.append(ch)
            else:
                if ch == quote_char:
                    in_quotes = False; quote_char = None; cur.append(ch)
                else:
                    cur.append(ch)
        elif ch == '[' and not in_quotes:
            bracket_depth += 1; cur.append(ch)
        elif ch == ']' and not in_quotes and bracket_depth > 0:
            bracket_depth -= 1; cur.append(ch)
        elif ch == ',' and not in_quotes and bracket_depth == 0:
            fields.append(''.join(cur).strip()); cur = []
        else:
            cur.append(ch)
    fields.append(''.join(cur).strip())
    return fields

def split_top_level_commas_inside_brackets(s: str):
    """
    Rep un string que COMENÇA amb '[' i acaba amb ']'.
    Retorna una llista d'ítems tallant per comes de nivell 1, ignorant cometes completament.
    """
    inner = s.strip()[1:-1]
    items, cur = [], []
    depth_square = 0
    depth_round  = 0
    depth_curly  = 0
    esc = False
    for ch in inner:
        if esc:
            cur.append(ch); esc = False; continue
        if ch == "\\":
            esc = True; continue
        if ch == "[":
            depth_square += 1; cur.append(ch); continue
        if ch == "]" and depth_square > 0:
            depth_square -= 1; cur.append(ch); continue
        if ch == "(":
            depth_round += 1; cur.append(ch); continue
        if ch == ")" and depth_round > 0:
            depth_round -= 1; cur.append(ch); continue
        if ch == "{":
            depth_curly += 1; cur.append(ch); continue
        if ch == "}" and depth_curly > 0:
            depth_curly -= 1; cur.append(ch); continue
        if ch == "," and depth_square == depth_round == depth_curly == 0:
            items.append(''.join(cur).strip()); cur = []; continue
        cur.append(ch)
    items.append(''.join(cur).strip())
    # neteja cada ítem: treu cometes EXTERIORS si n'hi ha (deixa apòstrofs interns intactes)
    cleaned = []
    for t in items:
        t = _strip_curly_quotes(t).strip()
        t = _strip_wrapping_quotes(t).strip()
        cleaned.append(t)
    return [x for x in cleaned if x != ""]

def parse_list_cell(s, fallback_empty=False):
    """
    Converteix cel·la a llista:
    - "['a','b']" o ['a','b'] -> ['a','b'] (amb apòstrofs interns OK)
    - a,b -> ['a','b']
    - val únic -> ['val']
    """
    if s is None:
        return [] if fallback_empty else s
    s = _strip_curly_quotes(str(s)).strip()
    s = _strip_wrapping_quotes(s).strip()
    if s == "":
        return [] if fallback_empty else []
    if s.startswith("[") and s.endswith("]"):
        try:
            # No confiem en cometes equilibrades: tallem per comes top-level
            return split_top_level_commas_inside_brackets(s)
        except Exception:
            pass
    if "," in s and "[" not in s and "]" not in s:
        return [x.strip().strip("'").strip('"') for x in s.split(",") if x.strip()]
    return [s.strip().strip("'").strip('"')]

def map_te_ordre(val: str) -> str:
    v = (val or "").strip().lower()
    if v.startswith("postre"):   return "ordre-postres"
    if v.startswith("primer"):   return "ordre-primer"
    if v.startswith("segon"):    return "ordre-segon"
    return "ordre-postres"

def map_apte_esdeveniment(list_val):
    lst = [str(x).strip() for x in (list_val or [])]
    if any(x.lower() == "totes" for x in lst):         return "tots"
    if any(x.lower() == "casament_only" for x in lst): return "casament_only"
    return "tots" if lst else "tots"

# ======= Llegeix CSV a mà =======
lines = Path(CSV_PATH).read_text(encoding="utf-8").splitlines()
if not lines:
    raise RuntimeError("El fitxer CSV és buit o no s'ha trobat.")

header_cols = [h.strip() for h in robust_split_csv_line(lines[0])]
lower_map = {h.lower(): i for i, h in enumerate(header_cols)}

def col(name):
    idx = lower_map.get(name.lower())
    if idx is None:
        raise RuntimeError(f"Falta la columna requerida: {name}")
    return idx

IDX_nom   = col("nom_plat")
IDX_mida  = col("mida_racio")
IDX_comp  = col("complexitat")
IDX_temp  = col("temperatura")
IDX_ordre = col("te_ordre")
IDX_form  = col("formalitat")
IDX_preu  = col("preu_cost")
IDX_apte  = col("apte_esdeveniment")
IDX_disp  = col("disponibilitat")
IDX_ing   = col("ingredients")
IDX_cat   = col("categoria")
IDX_proc  = col("procedencia_plat")

data_rows = [robust_split_csv_line(ln) for ln in lines[1:] if ln.strip()]
good_rows = [r for r in data_rows if len(r) >= len(header_cols)]
bad_rows  = [r for r in data_rows if len(r) < len(header_cols)]
if bad_rows:
    print(f"[AVÍS] {len(bad_rows)} files amb menys columnes. S'ignoren.")

# ======= Genera CLIPS =======
instances = []
seen_ids = set()
SEASONS_ORDER = ["estiu","hivern","primavera","tardor"]

for r in good_rows:
    def g(idx): return _strip_curly_quotes(r[idx]) if idx < len(r) else ""

    nom = g(IDX_nom).strip()
    if not nom:
        continue

    inst_id_base = slugify_instance_name(nom)
    inst_id = inst_id_base
    n = 2
    while inst_id in seen_ids:
        inst_id = f"{inst_id_base}-{n}"; n += 1
    seen_ids.add(inst_id)

    formalitat  = (g(IDX_form).strip() or "informal")
    temperatura = (g(IDX_temp).strip() or "tebi")

    complexitat = (g(IDX_comp).strip().lower() or "mitjana")
    if complexitat not in {"baixa","mitjana","alta"}: complexitat = "mitjana"

    mida = (g(IDX_mida).strip().lower() or "mitjana")
    if mida not in {"petita","mitjana","gran","alta"}: mida = "mitjana"

    te_ordre = map_te_ordre(g(IDX_ordre))

    apte = map_apte_esdeveniment(parse_list_cell(g(IDX_apte), fallback_empty=True))

    dispo_list = parse_list_cell(g(IDX_disp), fallback_empty=True)
    dispo_symbols = [x.strip().lower() for x in dispo_list if x.strip().lower() in {"estiu","hivern","primavera","tardor"}]
    if not dispo_symbols:
        dispo_symbols = SEASONS_ORDER.copy()
    else:
        order = {s:i for i,s in enumerate(SEASONS_ORDER)}
        dispo_symbols = sorted(set(dispo_symbols), key=lambda s: order.get(s, 99))

        ing_raw = parse_list_cell(g(IDX_ing), fallback_empty=True)

        # normalitza cada ingredient (treu accents, minúscules, espais) i elimina duplicats preservant l'ordre
        seen_ing = set()
        ing_norm = []
        for x in ing_raw:
            if not str(x).strip():
                continue
            t = normalize_ingredient_token(x)
            if t not in seen_ing:
                seen_ing.add(t)
                ing_norm.append(t)

        # escapa cometes per CLIPS i munta la llista final
        ing_list = [f"\"{sanitize_text(x)}\"" for x in ing_norm]

    categoria_sym = to_symbol(g(IDX_cat).strip() or "pastisseria")
    procedencia = sanitize_text(g(IDX_proc).strip() or "Internacional")

    preu_raw = _strip_wrapping_quotes(g(IDX_preu)).replace(",", ".").strip()
    try:
        preu = float(preu_raw)
    except Exception:
        preu = 0.0

    inst_lines = []
    inst_lines.append(f"  ({inst_id} of Plat")
    inst_lines.append(f"    (nom \"{sanitize_text(nom)}\")")
    inst_lines.append(f"    (formalitat \"{sanitize_text(formalitat)}\")")
    inst_lines.append(f"    (temperatura \"{sanitize_text(temperatura)}\")")
    inst_lines.append(f"    (complexitat {complexitat})")
    inst_lines.append(f"    (mida_racio {mida})")
    inst_lines.append(f"    (te_ordre {te_ordre})")
    inst_lines.append(f"    (apte_esdeveniment {apte})")
    inst_lines.append(f"    (preu_cost {preu:.1f})")
    inst_lines.append(f"    (disponibilitat_plats {' '.join(dispo_symbols)})")
    if ing_list:
        inst_lines.append(f"    (te_ingredients_noms {' '.join(ing_list)})")
    else:
        inst_lines.append(f"    (te_ingredients_noms)")
    inst_lines.append(f"    (categoria {categoria_sym})")
    inst_lines.append(f"    (procedencia_plat \"{procedencia}\")")
    inst_lines.append(f"  )")
    instances.append("\n".join(inst_lines))

header = f"(definstances {CLIPS_BLOCK_NAME}\n"
footer = ")\n"
content = header + "\n\n".join(instances) + "\n" + footer
Path(OUT_PATH).write_text(content, encoding="utf-8")

print(f"S'han generat {len(instances)} instàncies correctament.")
print(f"Fitxer CLIPS: {Path(OUT_PATH).resolve()}")
print("\n--- Vista prèvia ---")
print("\n".join(content.splitlines()[:120]))
